# M.Tech (AIML) Machine Learning - Assignment 2
## Multi-Model Classification & Evaluation on Breast Cancer Wisconsin (Diagnostic) Dataset
**Models Implemented**:
1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbor Classifier (KNN)
4. Gaussian Naive Bayes Classifier
5. Support Vector Machine (SVM)

In [1]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
)

In [2]:
RANDOM_SEED = 2026
raw_bundle = load_breast_cancer()
feature_names = [
    col.replace(" ", "_").replace("(", "").replace(")", "")
    for col in raw_bundle.feature_names
]
X_df = pd.DataFrame(raw_bundle.data, columns=feature_names)
y_series = pd.Series(raw_bundle.target, name="diagnosis_target")

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_series, test_size=0.25, random_state=RANDOM_SEED, stratify=y_series
)
print(f"Dataset Shape: {X_df.shape} | Target Shape: {y_series.shape}")
print(f"Training Set: {len(X_train)} instances | Test Set: {len(X_test)} instances")

Dataset Shape: (569, 30) | Target Shape: (569,)
Training Set: 426 instances | Test Set: 143 instances


In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(C=0.85, max_iter=1000, solver="lbfgs", random_state=RANDOM_SEED),
    "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=6, random_state=RANDOM_SEED),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance", metric="minkowski", p=2),
    "Naive Bayes": GaussianNB(var_smoothing=1e-8),
    "Support Vector Machine": SVC(C=1.2, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_SEED),
}

results = []
for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    results.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_pred if y_prob is None else y_prob), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9790 0.9958     0.9780  0.9889 0.9834 0.9550
         Decision Tree    0.9091 0.8991     0.9326  0.9222 0.9274 0.8060
                   KNN    0.9650 0.9828     0.9570  0.9889 0.9727 0.9251
           Naive Bayes    0.9021 0.9799     0.9419  0.9000 0.9205 0.7947
Support Vector Machine    0.9790 0.9962     0.9888  0.9778 0.9832 0.9553
